# 10D single integrator — comparing 5 controllers
## N=10 (1 CLF + 9 CBF), m=10

Obstacles are placed roughly halfway between each IC and the origin, so trajectories have to go around them rather than straight through.

Same NN for all N values — unused constraint slots get dummy padding (Remark 4.2 of the paper).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'cvxpy', '-q'], check=False)

import os, time, warnings
import numpy as np
import torch
import torch.nn as nn
import joblib
import cvxpy as cp
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| cvxpy:', cp.__version__)


In [ ]:
INPUT_DIR = "."
WORK_DIR  = "."

MODEL_PATH         = "/kaggle/input/datasets/bernatcot/resuldosos/best_kf_model_N10_m10.pt"
FT_HEAD_PATH       = "/kaggle/input/datasets/bernatcot/resuldosos/best_kf_model_N10_m10_ft.pt"
INPUT_SCALER_PATH  = "/kaggle/input/datasets/bernatcot/resuldosos/scaler_X_N10_m10.pkl"
OUTPUT_SCALER_PATH = "/kaggle/input/datasets/bernatcot/resuldosos/scaler_y_N10_m10.pkl"
print("Backbone :", MODEL_PATH)
print("FT head  :", FT_HEAD_PATH)
print("ScalerX  :", INPUT_SCALER_PATH)
print("ScalerY  :", OUTPUT_SCALER_PATH)


In [ ]:
# system dimensions
N_CONSTRAINTS = 10   # 1 CLF + 9 CBF
N_TRAIN       = 10
M_DIM_NN      = 10
STATE_DIM     = 10
IN_DIM_NN     = N_TRAIN * (1 + M_DIM_NN) + 1   # 111
DEVICE        = torch.device('cpu')

DT          = 0.005
T_MAX       = 20.0
STOP_RADIUS = 0.2

INITIAL_CONDITIONS = [
    [ 4.1,  -0.2,  1.3, -1.2,  2.1,  0.2,  1.1, -2.2,  0.4, -0.4],
    [-4.3,  1.1, -1.2,  2.2,  0.2, -1.3, -3.1,  1.4, -1.6,  1.1],
    [ 0.2,  4.1,  2.1, -1.1, -2.05,  0.4,  0.2,  3.2,  1.1, -1.2],
    [ 3.1, -3.1,  0.1,  1.2,  1.07, -2.1,  2.2, -1.1, -0.3,  0.4],
    [-2.2, -4.2,  1.66, -1.6,  0.4,  1.55, -1.4, -2.4,  0.2,  2.5],
    [ 2.05,  3.2, -2.21,  0.1, -1.06,  2.1,  1.7,  1.2, -1.1, -1.6],
    [-3.3,  2.2,  0.4,  2.1, -0.64, -1.2, -2.2,  0.3,  2.2,  1.7],
    [ 1.5, -4.6, -1.2,  1.2,  2.21,  0.1,  3.1, -3.2, -1.1,  0.2],
]

# obstacle centers are at ~45% along the IC->origin line
# radii all 0.8, verified no IC or origin is inside any sphere
OBSTACLES = [
    (np.array([2.2000, 0.0000, 0.5500, -0.5500, 1.1000, 0.0000, 0.5500, -1.1000, 0.2750, -0.2750], dtype=float), 0.8),
    (np.array([-2.4000, 0.6000, -0.6000, 1.2000, 0.0000, -0.6000, -1.8000, 0.6000, -0.9000, 0.6000], dtype=float), 0.8),
    (np.array([0.0000, 2.0000, 1.0000, -0.5000, -1.0000, 0.2500, 0.0000, 1.5000, 0.5000, -0.5000], dtype=float), 0.8),
    (np.array([1.7400, -1.7400, 0.0000, 0.5800, 0.5800, -1.1600, 1.1600, -0.5800, -0.2900, 0.2900], dtype=float), 0.8),
    (np.array([-1.0400, -2.0800, 0.7800, -0.7800, 0.2600, 0.7800, -0.5200, -1.0400, 0.0000, 1.0400], dtype=float), 0.8),
    (np.array([1.2400, 1.8600, -1.2400, 0.0000, -0.6200, 1.2400, 0.9300, 0.9300, -0.6200, -0.6200], dtype=float), 0.8),
    (np.array([-1.6800, 1.1200, 0.2800, 1.1200, -0.2800, -0.5600, -1.1200, 0.0000, 1.1200, 0.5600], dtype=float), 0.8),
    (np.array([0.5400, -2.1600, -0.5400, 0.5400, 1.0800, 0.0000, 1.6200, -1.6200, -0.5400, 0.0000], dtype=float), 0.8),
    (np.array([1.3750, 0.0000, -0.5500, 0.2750, 0.0000, 0.0000, 0.9625, 0.1375, -0.4125, -0.1375], dtype=float), 0.8),
]
assert len(OBSTACLES) == N_CONSTRAINTS - 1

_centers_arr = np.array([c for c, r in OBSTACLES], dtype=np.float64)
_radii_arr   = np.array([r for c, r in OBSTACLES], dtype=np.float64)
_A_arr = np.zeros(N_CONSTRAINTS, dtype=np.float64)
_B_arr = np.zeros((N_CONSTRAINTS, M_DIM_NN), dtype=np.float64)

_nn_vec = np.zeros(IN_DIM_NN, dtype=np.float32)
for _i in range(N_CONSTRAINTS, N_TRAIN):
    _nn_vec[_i * (M_DIM_NN + 1)] = -1.0

print(f'N_CONSTRAINTS={N_CONSTRAINTS}  IN_DIM_NN={IN_DIM_NN}')
print(f'ICs: {len(INITIAL_CONDITIONS)}  |  Obstacles: {len(OBSTACLES)}')

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.Dropout(dropout),
            nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.Dropout(dropout),
        )
    def forward(self, x): return x + self.block(x)

class KfNet(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256, n_blocks=5, dropout=0.15):
        super().__init__()
        self.stem   = nn.Linear(in_dim, hidden)
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head   = nn.Sequential(nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, out_dim))
    def forward(self, x): return self.head(self.blocks(self.stem(x)))

class KfNetFineTune(nn.Module):
    # backbone is frozen, only ft_head gets trained
    def __init__(self, pretrained, hidden=256, out_dim=10, adapter_dim=128, dropout=0.15):
        super().__init__()
        self.backbone = nn.Sequential(pretrained.stem, pretrained.blocks)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.ft_head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Linear(hidden, adapter_dim),
            nn.Dropout(dropout),
            nn.LayerNorm(adapter_dim),
            nn.SiLU(),
            nn.Linear(adapter_dim, out_dim),
        )
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.ft_head(features)

_base = KfNet(in_dim=IN_DIM_NN, out_dim=M_DIM_NN, hidden=256, n_blocks=5, dropout=0.15)
_base.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

model = KfNetFineTune(_base, hidden=256, out_dim=M_DIM_NN, adapter_dim=128).to(DEVICE)
model.ft_head.load_state_dict(torch.load(FT_HEAD_PATH, map_location=DEVICE))
model.eval()

scaler_X = joblib.load(INPUT_SCALER_PATH)
scaler_y = joblib.load(OUTPUT_SCALER_PATH)

# precompute scaler stats as tensors to avoid sklearn overhead at inference time
_mean_X  = torch.tensor(scaler_X.mean_,  dtype=torch.float32)
_std_X   = torch.tensor(scaler_X.scale_, dtype=torch.float32)
_mean_y  = torch.tensor(scaler_y.mean_,  dtype=torch.float32)
_std_y   = torch.tensor(scaler_y.scale_, dtype=torch.float32)

_q_tensor = torch.zeros(1, IN_DIM_NN, dtype=torch.float32)

try:
    model = torch.compile(model)
    print("torch.compile ok")
except Exception:
    print("torch.compile not available")

def nn_forward(q_raw):
    _q_tensor[0] = torch.from_numpy(np.asarray(q_raw, dtype=np.float32))
    x_scaled = (_q_tensor - _mean_X) / _std_X
    with torch.no_grad():
        z = model(x_scaled)
    return (z * _std_y + _mean_y).numpy()[0]

_dummy_q = np.zeros(IN_DIM_NN, dtype=np.float32)
for _ in range(20):
    nn_forward(_dummy_q)
print(f"model loaded, IN_DIM={IN_DIM_NN}")

In [ ]:
# single integrator: x_dot = u, so f=0, g=I
#
# CLF: V = 0.5*||x||^2,  W(x) = C_CLF*||x||^2
#   constraint: x^T u + C_CLF*||x||^2 <= 0
#   => a_clf = C_CLF*||x||^2,  b_clf = x
#
# CBF: reciprocal barrier h_i = CBF_GAIN*(1 - R_i^2/||x-c_i||^2)
#   safe set is h_i >= 0 (outside the sphere)
#   constraint from h_dot + h >= 0:
#   => a_i = -h_i,  b_i = -grad h_i

C_CLF    = 0.2
CBF_GAIN = 8.0

def gather_constraints(x):
    _A_arr[0]    = C_CLF * float(np.dot(x, x))
    _B_arr[0, :] = x

    diff    = x - _centers_arr
    dist_sq = np.maximum((diff * diff).sum(axis=1), 1e-8)
    r2      = _radii_arr ** 2

    h_vals = (dist_sq - r2).tolist() 

    h_cbf  = CBF_GAIN * (1.0 - r2 / dist_sq)
    grad_h = CBF_GAIN * (2.0 * r2 / dist_sq**2)
    _A_arr[1:]  = -h_cbf
    _B_arr[1:]  = -(grad_h[:, None] * diff)

    return list(_A_arr), [_B_arr[i].copy() for i in range(N_CONSTRAINTS)], h_vals

def scale_params(A_list, B_list):
    # Lemma 4.1 scaling
    M = max(max(abs(a) for a in A_list),
            max(np.linalg.norm(b) for b in B_list), 1.0)
    return [a/M for a in A_list], [b/M for b in B_list], 1.0/(M*M)

def build_nn_input(A_t, B_t, r_s):
    for i in range(N_CONSTRAINTS):
        base = i * (M_DIM_NN + 1)
        _nn_vec[base]                    = A_t[i]
        _nn_vec[base+1:base+1+M_DIM_NN] = B_t[i]
    a_dummy = float(-np.sqrt(r_s))
    for i in range(N_CONSTRAINTS, N_TRAIN):
        _nn_vec[i * (M_DIM_NN + 1)] = a_dummy
    _nn_vec[-1] = float(r_s)
    return _nn_vec

print(f"C_CLF={C_CLF}  CBF_GAIN={CBF_GAIN}")

In [ ]:
# HardNet-Aff projection (Min & Azizan 2025, Eq. 5)
#
# P(k) = k - A^T (A A^T + eps*I)^{-1} ReLU(A k + a)
#
# where A has rows b_i^T and a has entries a_i.
# The eps*I regularization handles near-singularity near x=0
# (b_clf = x -> 0 as x -> 0 makes A A^T ill-conditioned otherwise).

_hn_total_steps = 0
_hn_viol_steps  = 0
_hn_max_viol    = 0.0

def hardnet_project(k_nn, A_list, B_list):
    k     = np.asarray(k_nn, dtype=np.float64)
    A_mat = np.array(B_list, dtype=np.float64)
    a_vec = np.array(A_list, dtype=np.float64)

    v = A_mat @ k + a_vec
    if np.all(v <= 0.0):
        return k.copy()

    relu_v = np.maximum(v, 0.0)
    G      = A_mat @ A_mat.T
    lam    = np.linalg.solve(G + 0.01 * np.eye(len(G)), relu_v)
    return k - A_mat.T @ lam

print(f"HardNet defined  (G shape: {N_CONSTRAINTS}x{N_CONSTRAINTS})")


In [ ]:
# u* controller helpers
#
# cost function (Lemma 4.1, scaled form):
#   J_tilde(k) = -sum_i (||B_i||^2 + r*||k||^2) / (2*(A_i + B_i^T k))
#
# gradient:
#   grad J_tilde = sum_i  [ -r*k/d_i  +  (||B_i||^2 + r*||k||^2)*B_i / (2*d_i^2) ]
#   with d_i = A_i + B_i^T k  (negative inside K_p)
#
# we run gradient flow  k_dot = -grad J_tilde  until ||grad|| < GRAD_TOL

def grad_Jtilde(k, A_t, B_t, r):
    g    = np.zeros(M_DIM_NN, dtype=np.float64)
    k_sq = np.dot(k, k)
    for i in range(N_CONSTRAINTS):
        d = A_t[i] + np.dot(B_t[i], k)
        if d >= -1e-14:
            continue
        num = np.dot(B_t[i], B_t[i]) + r * k_sq
        g  += -(r * k) / d + (num / (2.0 * d * d)) * B_t[i]
    return g

def make_feasible(k, A_list, B_list, n_iter=500):
    # push k into K_p via iterated halfspace projections
    k = k.copy().astype(np.float64)
    for _ in range(n_iter):
        ok = True
        for i in range(N_CONSTRAINTS):
            viol = A_list[i] + np.dot(B_list[i], k)
            if viol > -1e-9:
                ok    = False
                nb_sq = np.dot(B_list[i], B_list[i])
                if nb_sq > 1e-14:
                    k -= (B_list[i] / nb_sq) * (viol + 1e-7)
        if ok:
            break
    return k

GRAD_TOL    = 1e-2
ODE_RTOL    = 1e-4
ODE_ATOL    = 1e-4
ODE_TMAX    = 2.0
ODE_MAXSTEP = np.inf

print(f"GRAD_TOL={GRAD_TOL}  ODE_RTOL={ODE_RTOL}  ODE_TMAX={ODE_TMAX}")

In [ ]:
# CLF-CBF QP: min ||u||^2  s.t. a_i + b_i^T u <= 0
_u_qp    = cp.Variable(M_DIM_NN)
_a_qp    = [cp.Parameter()         for _ in range(N_CONSTRAINTS)]
_b_qp    = [cp.Parameter(M_DIM_NN) for _ in range(N_CONSTRAINTS)]
_qp_prob = cp.Problem(
    cp.Minimize(cp.sum_squares(_u_qp)),
    [_a_qp[i] + _b_qp[i] @ _u_qp <= 0 for i in range(N_CONSTRAINTS)]
)

def solve_qp(A_list, B_list):
    for i in range(N_CONSTRAINTS):
        _a_qp[i].value = A_list[i]
        _b_qp[i].value = B_list[i]
    try:
        _qp_prob.solve(solver=cp.OSQP, warm_starting=True,
                       eps_abs=1e-6, eps_rel=1e-6, max_iter=10000, verbose=False)
        if _u_qp.value is not None:
            return _u_qp.value.copy()
    except Exception:
        pass
    return np.zeros(M_DIM_NN)

print(f"QP set up  (N={N_CONSTRAINTS}, m={M_DIM_NN})")

In [ ]:
def simulate(x0, control_fn):
    N_steps   = int(T_MAX / DT) + 1
    n_cbf     = len(OBSTACLES)
    traj      = np.zeros((N_steps, STATE_DIM))
    h_hist    = np.zeros((N_steps, n_cbf))
    V_hist    = np.zeros(N_steps)
    traj[0]   = x0
    V_hist[0] = 0.5 * np.dot(x0, x0)
    u0, t0, h0 = control_fn(x0)
    h_hist[0]  = h0
    times      = [t0]
    for i in range(1, N_steps):
        x = traj[i-1]
        if np.linalg.norm(x) < STOP_RADIUS:
            traj[i:]   = x
            h_hist[i:] = h_hist[i-1]
            V_hist[i:] = V_hist[i-1]
            break
        u, tc, hv = control_fn(x)
        traj[i]   = x + u * DT
        h_hist[i] = hv
        V_hist[i] = 0.5 * np.dot(traj[i], traj[i])
        times.append(tc)
    return traj, h_hist, V_hist, np.array(times)

def run_all_ics(control_fn, label):
    all_times, trajs, all_h, all_V = [], [], [], []
    print(f"\nSimulating: {label}")
    for idx, ic in enumerate(INITIAL_CONDITIONS):
        x0 = np.array(ic, dtype=float)
        traj, h_hist, V_hist, step_times = simulate(x0, control_fn)
        trajs.append(traj); all_h.append(h_hist)
        all_V.append(V_hist); all_times.append(step_times)
        V_last = V_hist[V_hist > 0][-1] if (V_hist > 0).any() else 0.0
        print(f"  IC {idx+1}/{len(INITIAL_CONDITIONS)}  "
              f"steps={len(step_times)}  min_h={h_hist.min():.3f}  V_final={V_last:.4f}")
    flat = np.concatenate(all_times) * 1e3
    print(f"  time (ms): mean={flat.mean():.4f}  std={flat.std():.4f}  "
          f"median={np.median(flat):.4f}  p95={np.percentile(flat, 95):.4f}")
    return trajs, all_h, all_V, flat

print("simulation ready")

In [ ]:
def plot_all(trajs, all_h, all_V, label, prefix):
    n_cbf  = len(OBSTACLES)
    colors = plt.cm.tab10(np.linspace(0, 0.85, len(trajs)))
    t_arr  = np.arange(int(T_MAX / DT) + 1) * DT
    slices = [(0, 1), (2, 3), (4, 5), (6, 7)]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    short_label = label.split(": ", 1)[-1] if ": " in label else label
    fig.suptitle(f"{short_label}\n10D Trajectories", fontsize=22, fontweight="bold")
    for ax, (d1, d2) in zip(axes.flatten(), slices):
        ax.set_title(f"$x_{{{d1+1}}}$ - $x_{{{d2+1}}}$", fontsize=20)
        for traj, col in zip(trajs, colors):
            ax.plot(traj[:, d1], traj[:, d2], color=col, alpha=0.75, lw=1.8)
            ax.scatter(traj[0, d1], traj[0, d2], color=col, s=60, zorder=6, edgecolors="k", lw=0.5)
        ax.scatter(0, 0, s=60, marker="o", color="k", zorder=10)  
        for c, r in OBSTACLES:
            circ = plt.Circle((c[d1], c[d2]), r, color="tomato", alpha=0.20)
            ax.add_patch(circ)
            ax.plot(c[d1], c[d2], "r+", ms=8, mew=1.5)
        ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect("equal")
        ax.axhline(0, color="k", lw=0.8, alpha=0.7)
        ax.axvline(0, color="k", lw=0.8, alpha=0.7)
        ax.grid(True, ls="--", lw=0.35, alpha=0.5)
        ax.tick_params(labelsize=15)
        ax.set_xlabel(f"$x_{{{d1+1}}}$", fontsize=18)
        ax.set_ylabel(f"$x_{{{d2+1}}}$", fontsize=18)
    plt.tight_layout()
    p = os.path.join(WORK_DIR, f"{prefix}_traj.pdf") 
    plt.savefig(p, bbox_inches="tight")
    print(f"  Saved: {p}")
    plt.show()

    # Lyapunov
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (V_hist, col) in enumerate(zip(all_V, colors)):
        mask = V_hist > 0
        n    = np.where(mask)[0][-1] + 2 if mask.any() else len(V_hist)
        ax.semilogy(t_arr[:n], V_hist[:n], color=col, lw=2.2, alpha=0.85, label=f"IC {i+1}")
    ax.set_xlabel("Time (s)", fontsize=18)
    ax.set_ylabel("$V(x) = 0.5\\|x\\|^2$", fontsize=18)
    ax.set_title(f"{short_label} - Lyapunov", fontsize=20, fontweight="bold")
    ax.tick_params(labelsize=15)
    ax.legend(ncol=4, fontsize=13) 
    ax.grid(True, ls="--", lw=0.35, alpha=0.5)
    plt.tight_layout()
    p = os.path.join(WORK_DIR, f"{prefix}_lyapunov.pdf") 
    plt.savefig(p, bbox_inches="tight")
    print(f"  Saved: {p}")
    plt.show()

    # individual CBF values
    ncols = min(n_cbf, 3)
    nrows = (n_cbf + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes_flat = np.array(axes).flatten() if n_cbf > 1 else [axes]
    fig.suptitle(f"{short_label} - CBF values", fontsize=22, fontweight="bold")
    for obs_idx in range(n_cbf):
        ax = axes_flat[obs_idx]
        for i, (h_hist, col) in enumerate(zip(all_h, colors)):
            ax.plot(t_arr[:h_hist.shape[0]], h_hist[:, obs_idx], color=col, lw=1.5, alpha=0.75)
        ax.axhline(0, color="red", lw=2.0, ls="--", alpha=0.9)
        ax.fill_between(t_arr[:all_h[0].shape[0]], -50, 0, alpha=0.04, color="red")
        ax.set_title(f"", fontsize=18)
        ax.set_xlabel("Time (s)", fontsize=18)
        ax.set_ylabel(f"$h_{{{obs_idx+1}}}(x)$", fontsize=18) 
        ax.tick_params(labelsize=15)
        ax.grid(True, ls="--", lw=0.35, alpha=0.5)
        ax.set_xlim(0, T_MAX)
    for j in range(n_cbf, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.tight_layout()
    p = os.path.join(WORK_DIR, f"{prefix}_cbf.pdf") 
    plt.savefig(p, bbox_inches="tight")
    print(f"  Saved: {p}")
    plt.show()

    # min CBF safety margin
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (h_hist, col) in enumerate(zip(all_h, colors)):
        ax.plot(t_arr[:h_hist.shape[0]], h_hist.min(axis=1),
                color=col, lw=2.2, alpha=0.85, label=f"IC {i+1}")
    ax.fill_between(t_arr[:all_h[0].shape[0]], -50, 0, alpha=0.05, color="red")
    ax.set_xlabel("Time (s)", fontsize=18)
    ax.set_ylabel("$\\min_i\\, h_i(x)$", fontsize=18)
    ax.set_title(f"{short_label} - Safety Margin", fontsize=20, fontweight="bold")
    ax.tick_params(labelsize=15)
    ymin = min(h.min() for h in all_h) - 0.5
    ax.set_ylim(bottom=min(ymin, -1.0)); ax.set_xlim(0, T_MAX)
    ax.legend(ncol=4, fontsize=13) 
    ax.grid(True, ls="--", lw=0.35, alpha=0.5)
    plt.tight_layout()
    p = os.path.join(WORK_DIR, f"{prefix}_min_cbf.pdf") 
    plt.savefig(p, bbox_inches="tight")
    print(f"  Saved: {p}")
    plt.show()

print("plot_all ready")

## Controller 1 - Pure NN

Direct NN output $k_{\rm nn}$

In [ ]:
def ctrl_nn(x):
    t0 = time.perf_counter()
    A_list, B_list, h_vals = gather_constraints(x)
    A_t, B_t, r_s = scale_params(A_list, B_list)
    nn_in = build_nn_input(A_t, B_t, r_s)
    u     = nn_forward(nn_in)
    return u, time.perf_counter() - t0, h_vals

trajs_nn, all_h_nn, all_V_nn, times_nn = run_all_ics(ctrl_nn, "Controller 1: NN")
np.save(os.path.join(WORK_DIR, "timings_nn.npy"), times_nn)
plot_all(trajs_nn, all_h_nn, all_V_nn, "Pure NN", "ctrl1_nn")


## Controller 2 - HardNet-Aff  (Min & Azizan 2025)

$P(k) = k - A^\top(AA^\top)^{-1}\,\mathrm{ReLU}(Ak+a)$

In [ ]:
_hn_total_steps = 0
_hn_viol_steps  = 0
_hn_max_viol    = 0.0

def ctrl_hardnet(x):
    t0 = time.perf_counter()
    A_list, B_list, h_vals = gather_constraints(x)
    A_t, B_t, r_s = scale_params(A_list, B_list)
    nn_in = build_nn_input(A_t, B_t, r_s)
    k_nn  = nn_forward(nn_in)                         
    u     = hardnet_project(k_nn, A_list, B_list)      
    return u, time.perf_counter() - t0, h_vals

trajs_hn, all_h_hn, all_V_hn, times_hn = run_all_ics(ctrl_hardnet, "Controller 2: HardNet-Aff")
np.save(os.path.join(WORK_DIR, "timings_hardnet.npy"), times_hn)
plot_all(trajs_hn, all_h_hn, all_V_hn, "HardNet-Aff", "ctrl2_hardnet")


## Controller 3 - CLF-CBF QP 

$\min\|u\|^2$ s.t. $a_i+b_i^\top u \leq 0$, solved online.

In [ ]:
def ctrl_qp(x):
    t0 = time.perf_counter()  
    A_list, B_list, h_vals = gather_constraints(x)
    u  = solve_qp(A_list, B_list)
    return u, time.perf_counter() - t0, h_vals

trajs_qp, all_h_qp, all_V_qp, times_qp = run_all_ics(ctrl_qp, "Controller 3: CLF-CBF QP")
np.save(os.path.join(WORK_DIR, "timings_qp.npy"), times_qp)
plot_all(trajs_qp, all_h_qp, all_V_qp, "CLF-CBF QP", "ctrl3_qp")


## Controller 4 - $u^*$ with QP warmstart

Gradient-flow integration of $\tilde J_q$ warmstarted from QP.

In [ ]:
def ctrl_ustar_qp(x):
    A_list, B_list, h_vals = gather_constraints(x)
    A_t, B_t, r_s = scale_params(A_list, B_list)
    t0     = time.perf_counter()
    k_warm = solve_qp(A_list, B_list)
    k_warm = make_feasible(k_warm, A_list, B_list)

    def neg_grad(t, k):
        return -grad_Jtilde(k, A_t, B_t, r_s)

    def ev_conv(t, k):
        return np.linalg.norm(grad_Jtilde(k, A_t, B_t, r_s)) - GRAD_TOL
    ev_conv.terminal  = True
    ev_conv.direction = -1

    try:
        sol = solve_ivp(neg_grad, [0.0, ODE_TMAX], k_warm, method="RK45",
                        rtol=ODE_RTOL, atol=ODE_ATOL, events=ev_conv,
                        dense_output=False, max_step=ODE_MAXSTEP)
        u = sol.y[:, -1]
    except Exception:
        u = k_warm

    return u, time.perf_counter() - t0, h_vals

trajs_qpws, all_h_qpws, all_V_qpws, times_qpws = run_all_ics(ctrl_ustar_qp, "Controller 4: u* QP warmstart")
np.save(os.path.join(WORK_DIR, "timings_ustar_qp_warmstart.npy"), times_qpws)
plot_all(trajs_qpws, all_h_qpws, all_V_qpws, "u* QP warmstart", "ctrl4_ustar_qp")


## Controller 5 - $u^*$ with NN warmstart

Gradient-flow integration of $\tilde J_q$ warmstarted from NN.

In [ ]:
def ctrl_ustar_nn(x):
    A_list, B_list, h_vals = gather_constraints(x)
    A_t, B_t, r_s = scale_params(A_list, B_list)
    t0     = time.perf_counter()
    nn_in  = build_nn_input(A_t, B_t, r_s)
    k_warm = nn_forward(nn_in)             
    k_warm = make_feasible(k_warm, A_list, B_list)

    def neg_grad(t, k): return -grad_Jtilde(k, A_t, B_t, r_s)
    def ev_conv(t, k):  return np.linalg.norm(grad_Jtilde(k, A_t, B_t, r_s)) - GRAD_TOL
    ev_conv.terminal  = True
    ev_conv.direction = -1

    try:
        sol = solve_ivp(neg_grad, [0.0, ODE_TMAX], k_warm, method="RK45",
                        rtol=ODE_RTOL, atol=ODE_ATOL, events=ev_conv,
                        dense_output=False, max_step=ODE_MAXSTEP)
        u = sol.y[:, -1]
    except Exception:
        u = k_warm
    return u, time.perf_counter() - t0, h_vals

trajs_nnws, all_h_nnws, all_V_nnws, times_nnws = run_all_ics(ctrl_ustar_nn, "Controller 5: u* NN warmstart")
np.save(os.path.join(WORK_DIR, "timings_ustar_nn_warmstart.npy"), times_nnws)
plot_all(trajs_nnws, all_h_nnws, all_V_nnws, "u* NN warmstart", "ctrl5_ustar_nn")


## Table 1 -- Summary

In [ ]:
import csv
results = {
    "NN"                 : times_nn,
    "HardNet-Aff"        : times_hn,
    "CLF-CBF QP"         : times_qp,
    "u* w/ QP warmstart" : times_qpws,
    "u* w/ NN warmstart" : times_nnws,
}
all_h_map = {
    "NN"                 : all_h_nn,
    "HardNet-Aff"        : all_h_hn,
    "CLF-CBF QP"         : all_h_qp,
    "u* w/ QP warmstart" : all_h_qpws,
    "u* w/ NN warmstart" : all_h_nnws,
}
all_trajs_map = {
    "NN"                 : trajs_nn,
    "HardNet-Aff"        : trajs_hn,
    "CLF-CBF QP"         : trajs_qp,
    "u* w/ QP warmstart" : trajs_qpws,
    "u* w/ NN warmstart" : trajs_nnws,
}
print()
print("=" * 80)
print(f"  TABLE 1 -- Execution times (ms)  N={N_CONSTRAINTS}, m=10  (1 CLF + {N_CONSTRAINTS-1} CBF)")
print("=" * 80)
print(f"  {chr(32)+chr(32)+chr(32)+chr(32)+chr(32)+chr(32)+chr(32)}Controller{chr(32)*13}  {chr(32)*3}Mean  {chr(32)*4}Std  {chr(32)*2}Median  {chr(32)*4}p95  {chr(32)*5}N")
print("-" * 80)
for lbl, arr in results.items():
    print(f"  {lbl:<22}  {arr.mean():>8.4f}  {arr.std():>8.4f}  "
          f"{np.median(arr):>8.4f}  {np.percentile(arr,95):>8.4f}  {len(arr):>6}")
print("=" * 80)
print()
print("  Safety check  (min h_i(x) = ||x-c_i||^2 - R_i^2  across all ICs and steps):")
for lbl, all_h in all_h_map.items():
    min_h = min(h.min() for h in all_h)
    ok    = "SAFE" if min_h >= -1e-3 else "VIOLATED"
    print(f"    {lbl:<22}: min h = {min_h:+.4f}  {ok}")
print()
print("  Convergence check  (final ||x|| per IC):")
for lbl, trajs in all_trajs_map.items():
    finals = [np.linalg.norm(t[t.any(axis=1)][-1]) for t in trajs]
    print(f"    {lbl:<22}: " + "  ".join(f"{v:.3f}" for v in finals))
csv_path = os.path.join(WORK_DIR, "table1_N4_3cbf.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Controller","Mean_ms","Std_ms","Median_ms","p95_ms","N_steps","min_CBF"])
    for lbl, arr in results.items():
        min_h = min(h.min() for h in all_h_map[lbl])
        w.writerow([lbl, f"{arr.mean():.4f}", f"{arr.std():.4f}",
                    f"{np.median(arr):.4f}", f"{np.percentile(arr,95):.4f}",
                    len(arr), f"{min_h:.4f}"])
print(f"\nCSV saved: {csv_path}")


## Comparison - All 5 Controllers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
labels     = list(results.keys())
data       = [results[l] for l in labels]
short      = ["NN", "HardNet", "QP", "u* QP", "u* NN"]
colors_box = plt.cm.Set2(np.linspace(0, 0.9, 5))

ax = axes[0]
bp = ax.boxplot(data, labels=short, patch_artist=True, showfliers=False)
for patch, col in zip(bp["boxes"], colors_box):
    patch.set_facecolor(col); patch.set_alpha(0.75)
ax.set_ylabel("Execution time (ms)", fontsize=18)
ax.tick_params(labelsize=15)
ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.6)

ax = axes[1]
means = [results[l].mean() for l in labels]
stds  = [results[l].std()  for l in labels]
x_pos = np.arange(len(labels))
bars  = ax.bar(x_pos, means, yerr=stds, capsize=5,
               color=colors_box, alpha=0.8, edgecolor="k", lw=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(short, fontsize=15)
ax.set_ylabel("Mean time (ms)", fontsize=18)
ax.set_title("Mean \u00b1 std execution time", fontsize=20, fontweight="bold")
ax.tick_params(labelsize=15)
ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.6)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{mean:.3f}", ha="center", va="bottom", fontsize=13)

plt.tight_layout()
p = os.path.join(WORK_DIR, "table1_timing_comparison.pdf")
plt.savefig(p, bbox_inches="tight")
print(f"Saved: {p}")
plt.show()

# comparison: Lyapunov + safety margin across all controllers
fig, axes   = plt.subplots(1, 2, figsize=(16, 6))
ctrl_colors = plt.cm.Set1(np.linspace(0, 0.8, 5))
ctrl_labels = ["NN", "HardNet", "QP", "u* QP ws", "u* NN ws"]
all_V_list  = [all_V_nn, all_V_hn, all_V_qp, all_V_qpws, all_V_nnws]
all_h_list  = [all_h_nn, all_h_hn, all_h_qp, all_h_qpws, all_h_nnws]
t_arr       = np.arange(int(T_MAX / DT) + 1) * DT

ax = axes[0]
for col, lbl, all_V in zip(ctrl_colors, ctrl_labels, all_V_list):
    ax.semilogy(t_arr, np.mean(all_V, axis=0), color=col, lw=2.5, label=lbl)
ax.set_xlabel("Time (s)", fontsize=18)
ax.set_ylabel("Mean $V(x)$ (log scale)", fontsize=18)
ax.set_title("Lyapunov convergence", fontsize=20, fontweight="bold")
ax.tick_params(labelsize=15)
ax.legend(fontsize=15)
ax.grid(True, ls="--", lw=0.35, alpha=0.5)

ax = axes[1]
for col, lbl, all_h in zip(ctrl_colors, ctrl_labels, all_h_list):
    min_h_mean = np.mean([h.min(axis=1) for h in all_h], axis=0)
    ax.plot(t_arr[:len(min_h_mean)], min_h_mean, color=col, lw=2.5, label=lbl)
ax.set_xlabel("Time (s)", fontsize=18)
ax.set_ylabel("Mean $\\min_i\\, h_i(x)$", fontsize=18)
ax.set_title("Safety margin", fontsize=20, fontweight="bold")
ax.tick_params(labelsize=15)
ax.legend(fontsize=15, loc="upper right") 
ax.grid(True, ls="--", lw=0.35, alpha=0.5)

plt.tight_layout()
p = os.path.join(WORK_DIR, "comparison_V_and_safety.pdf")
plt.savefig(p, bbox_inches="tight")
print(f"Saved: {p}")
plt.show()